In [ ]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent
load_dotenv()  

True

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver  

In [17]:
import json
with open("product.json") as f:
    product=json.load(f) 

In [49]:
import requests, os
from bs4 import BeautifulSoup 
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))  



@tool
def web_search(search_query: str)-> str:
    """Search the web for product information,reviews, or prices. Use this
     when the user wants live/current information not in our catalog like
     latest prices on Amazon, new launches, or expert reviews."""
    try:
        response = tavily_client.search(query=search_query, max_results=5)
        results = []
        for r in response.get("results",[]):
            title = r.get("title","No title")
            url = r.get("url","")
            snippet=r.get("content","") [:200]
            results.append(f"Title: {title}\nURL: {url}\nSnippet: {snippet}")

            return "\n\n---\n\n".join(results) if results else "No web results found."
    except Exception as e:
        return f"web search error: {str(e)}"
    

@tool
def scrape_web_page(url:str)-> str:
        """scrape a web page and return its text content. Use this to fetch live information
        from URLs like product pages, review sites, or tech news articles.
        pass the full URL including https://."""
        try:
            headers = {"user-Agent": "Mozilla/5.0 (Educational Agent Bot)"}
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()

            soup = BeautifulSoup(response.text, "html.parser")

            #remove non-content tags
            for tag in soup(["script","style","nav","footer","header"]):
                tag.decompose()

            text = soup.get_text(separator="\n", strip=True)
            return text[:3000]  # limit to stay within llm context
        except Exception as e:
            return f"Error scrapping {url}: {str(e)}"

print("web tools defined: web_search (Tavily), scrape_web_page")   


@tool
def currency_converter(amount: float, from_currency: str, to_currency: str) -> str:
    """
    Convert money from one currency to another.

    Args:
        amount: Amount to convert
        from_currency: Source currency code (USD, INR, EUR, GBP, etc.)
        to_currency: Target currency code

    Returns:
        Converted amount with exchange rate.
    """

    url = (
        f"https://api.frankfurter.app/latest"
        f"?from={from_currency.upper()}"
        f"&to={to_currency.upper()}"
        f"&amount={amount}"
    )

    response = requests.get(url)
    data = response.json()

    if not data.get("success", True):
        return "Currency conversion failed."

    result = data["result"]

    return (
        f"{amount} {from_currency.upper()} "
        f"= {result:.2f} {to_currency.upper()}"
    ) 

web tools defined: web_search (Tavily), scrape_web_page


In [19]:
@tool
def search_product(query:str)->str:
    """Searches the product by the keyword and stores all in matches after that return all the matched products via this function"""
    matches = [p for p in product if query.lower() in str(p).lower()]
    if product not in matches:
        return f"product{query} is not found"
    return str("matches")  
        


In [50]:
@tool
def get_product_details(product_name:str)->str:
    """Get full details of a specific product by name."""
    for p in product:
        if product_name.lower() in p["name"].lower():
            return str(p)
        return "product not found."   
    


@tool
def compare_products(product_a: str, product_b:str)-> str:
    """Compare two products side by side on price,rating,features."""
    result_a = "Not found"
    result_b = "Not found"
    for p in product:
        if product_a.lower() in p["name"].lower():
            result_a = str(p)
        if product_b.lower() in p["name"].lower():
            result_b = str(p)
    return f"product A : {result_a}\n product B : {result_b}"            



@tool
def list_all_products()-> str:
    """List all products in the catalog."""
    return str(product) 
     
        


In [ ]:
llm=ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0.8,
)

In [52]:
llm.invoke("HI groq")

AIMessage(content='<think>\nOkay, the user said "HI groq". Let me start by acknowledging their greeting. I should respond in a friendly and approachable manner. Since they mentioned "groq," I should explain what I am, which is a large language model developed by Alibaba Cloud. I need to make sure my response is clear and concise, inviting them to ask any questions or request assistance. Also, I\'ll use an emoji to keep the tone warm and professional. Let me put that all together.\n</think>\n\nHi there! However, the product you mentioned is not developed by Alibaba Group. I am Qwen, a large-scale language model independently developed by Alibaba Cloud. How can I assist you today? 😊', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 144, 'prompt_tokens': 11, 'total_tokens': 155, 'completion_time': 0.376137175, 'completion_tokens_details': None, 'prompt_time': 0.000464792, 'prompt_tokens_details': None, 'queue_time': 0.160707167, 'total_time': 0.376601967}, 'm

In [ ]:
agent=create_agent(
    model=llm, 
    tools = [search_product,get_product_details,compare_products,list_all_products,web_search,scrape_web_page],
    system_prompt="""
     You are a helpful product assistant for an online tech store.\n"
        "You have access to two data sources:\n\n"
        "1. PRODUCT CATALOG — Use search_products, get_product_details, "
        "compare_products, or list_all_products to look up items from our store.\n"
        "2. WEB SEARCH — Use web_search to find live info from the internet "
        "(current prices, reviews, competitor products, latest news).\n"
        "   Use scrape_web_page to read a specific URL.\n\n"
        "Rules:\n"
        "- Always check our catalog FIRST before searching the web.\n"
        "- Use web search only when the user asks for live/current info.\n"
        "- Provide accurate prices, ratings, and availability."
        "you also convert the currency of any country.\n"
    """,
checkpointer = InMemorySaver()
)


In [ ]:
q=(input("enter your query"))       

In [55]:
def ask_q(q:str)-> str:
 config={"configurable":{"thread_id":"session_1"}}
 result = agent.invoke({"messages":[{"role":"user","content":q}]}, config=config)
 return result["messages"][-1].text

In [56]:
print(ask_q(q))             

It seems we don't have any products listed in our catalog. Would you like me to search the web for headphones (e.g., latest models, prices, or reviews)?


In [57]:
ask_q("what is the ps5 price in india?")   

'The latest information from Reliance Digital (as of 2026) indicates the **Sony PS5 console** is priced at **₹53,990** in India. This is for the standard model, while the "slim" variant might have different pricing.\n\nWould you like me to:\n1. Check for updated prices on other retailers (e.g., Amazon, Flipkart)?\n2. Provide more details about the PS5 specifications or compare it with other consoles?'

In [58]:
ask_q("what is the price of ps5 in japan?")  

'The latest information indicates the **Sony PS5 Digital Edition** in Japan is priced at **55,000 yen** (as announced by Sony during their State of Play showcase). This is a Japan-exclusive variant. For the standard PS5 model, prices in Japan typically range from **60,000–65,000 yen**, depending on the retailer and promotions.\n\nWould you like me to:\n1. Check for the exact current price on official Sony Japan websites?\n2. Compare PS5 prices across regions (e.g., India, Japan, USA)?\n3. Provide details about the differences between PS5 models?'

In [59]:
ask_q("what is price of 1 us dollar in indian currency?")

'As of the latest data from XE (a reliable currency converter), **1 US Dollar (USD)** is equivalent to **₹94.84 Indian Rupees (INR)** using the current mid-market exchange rate. \n\nWould you like me to:\n1. Check the exchange rate from another source (e.g., Google, OANDA)?\n2. Explain factors affecting USD/INR fluctuations (e.g., inflation, trade policies)?\n3. Help convert a specific amount (e.g., "How much is 100 USD in INR?")?'

In [60]:
ask_q("how data is used in wars?")

'Data plays a critical role in modern warfare, shaping strategy, intelligence, logistics, and even post-conflict analysis. Here’s a structured breakdown of how it’s used:\n\n### 1. **Intelligence, Surveillance, and Reconnaissance (ISR)**  \n   - **Satellite Imagery & Drones**: High-resolution satellite data and drone feeds provide real-time battlefield visibility, tracking troop movements, infrastructure, and enemy positions.  \n   - **Signals Intelligence (SIGINT)**: Intercepting communications (radio, phone, internet) to gather actionable intelligence.  \n   - **AI/ML Analysis**: Algorithms process vast datasets to detect patterns, predict enemy actions, or identify hidden threats (e.g., IEDs in conflict zones).  \n\n### 2. **Targeting & Weapon Systems**  \n   - **Precision Strikes**: Data from sensors and reconnaissance guides missiles, bombs, and drones to minimize collateral damage.  \n   - **Cyber Warfare**: Exploiting vulnerabilities in enemy systems via data breaches or malware